In [0]:
%pip install requests
%restart_python

In [0]:
from datetime import datetime, timezone
from pathlib import Path
import json
import time

import requests

In [0]:
CATALOG = "opensky_lakehouse"
LANDING_SCHEMA = "landing"
VOLUME = "raw_files"

RAW_BASE_PATH = f"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME}/opensky/states"

BOUNDING_BOX = {
    "lamin": 35.0,
    "lomin": -10.0,
    "lamax": 44.5,
    "lomax": 4.5,
}

REQUEST_TIMEOUT_SECONDS = 30

In [0]:
OPENSKY_CLIENT_ID = dbutils.secrets.get("opensky", "client_id")
OPENSKY_CLIENT_SECRET = dbutils.secrets.get("opensky", "client_secret")

In [0]:
def get_opensky_token(client_id: str, client_secret: str) -> str:
    token_url = (
        "https://auth.opensky-network.org/auth/realms/opensky-network/"
        "protocol/openid-connect/token"
    )

    response = requests.post(
        token_url,
        data={
            "grant_type": "client_credentials",
            "client_id": client_id,
            "client_secret": client_secret,
        },
        headers={
            "Content-Type": "application/x-www-form-urlencoded"
        },
        timeout=REQUEST_TIMEOUT_SECONDS,
    )

    response.raise_for_status()
    return response.json()["access_token"]

In [0]:
def fetch_opensky_states() -> dict:
    token = get_opensky_token(
        OPENSKY_CLIENT_ID,
        OPENSKY_CLIENT_SECRET,
    )

    response = requests.get(
        "https://opensky-network.org/api/states/all",
        params=BOUNDING_BOX,
        headers={
            "Authorization": f"Bearer {token}"
        },
        timeout=REQUEST_TIMEOUT_SECONDS,
    )

    response.raise_for_status()
    data = response.json()

    return {
        "source": "opensky",
        "retrieved_at": datetime.now(timezone.utc).isoformat(),
        "bounding_box": BOUNDING_BOX,
        "api_time": data.get("time"),
        "aircraft_count": len(data.get("states") or []),
        "states": data.get("states") or [],
    }

In [0]:
def save_raw_json(payload: dict) -> str:
    retrieved_at = datetime.fromisoformat(
        payload["retrieved_at"].replace("Z", "+00:00")
    )

    date_part = retrieved_at.strftime("%Y-%m-%d")
    timestamp_part = retrieved_at.strftime("%Y%m%dT%H%M%S")

    output_dir = Path(f"{RAW_BASE_PATH}/date={date_part}")
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / f"opensky_states_{timestamp_part}.json"

    with output_file.open("w", encoding="utf-8") as file:
        json.dump(payload, file, ensure_ascii=False)

    return str(output_file)

In [0]:
payload = fetch_opensky_states()
output_file = save_raw_json(payload)

print(f"Aviones recibidos: {payload['aircraft_count']}")
print(f"Timestamp API: {payload['api_time']}")
print(f"Archivo guardado en: {output_file}")

In [0]:
display(dbutils.fs.ls(RAW_BASE_PATH))

In [0]:
display(dbutils.fs.ls(f"{RAW_BASE_PATH}/date={datetime.now(timezone.utc).strftime('%Y-%m-%d')}"))